# Validación de audit logs y logrotate en Kubernetes

Este notebook comprueba el almacenamiento persistente de auditoría del clúster Vault PR, los montajes de los pods, el audit device de Vault y el sidecar de `logrotate`.

Las comprobaciones normales son de solo lectura. La última prueba fuerza una rotación únicamente si se cambia `FORCE_ROTATION=0` por `FORCE_ROTATION=1`.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = next(
    (directory / ".env" for directory in (Path.cwd(), *Path.cwd().parents) if (directory / ".env").is_file()),
    None,
)
if ENV_FILE:
    load_dotenv(ENV_FILE, override=True)

os.environ.setdefault("KUBE_CONTEXT", "PR")
os.environ.setdefault("VAULT_K8S_NAMESPACE", "vaultpr")
os.environ.setdefault("VAULT_HELM_RELEASE_NAME", "vaultpr")
os.environ.setdefault("VAULT_AUDIT_PATH", "/vault/audit/vault.log")

print(f"Contexto:  {os.environ['KUBE_CONTEXT']}")
print(f"Namespace: {os.environ['VAULT_K8S_NAMESPACE']}")
print(f"Release:   {os.environ['VAULT_HELM_RELEASE_NAME']}")
print(f".env:      {ENV_FILE or 'no encontrado (solo será necesario para consultar Vault)'}")

## 1. Contexto, StatefulSet y pods

Verifica que el contexto existe, que el StatefulSet está disponible y que cada pod incluye los contenedores `vault` y `auditlog-rotator`.

In [ ]:
%%bash
set -euo pipefail

kubectl config get-contexts "${KUBE_CONTEXT}"
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get statefulset "${VAULT_HELM_RELEASE_NAME}" -o wide
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get pods -l 'app.kubernetes.io/name=vault,component=server' \
  -o custom-columns='POD:.metadata.name,READY:.status.containerStatuses[*].ready,CONTAINERS:.spec.containers[*].name,RESTARTS:.status.containerStatuses[*].restartCount'

not_ready=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get pods -l 'app.kubernetes.io/name=vault,component=server' -o json | \
  jq '[.items[].status.containerStatuses[] | select(.ready != true)] | length')
test "${not_ready}" -eq 0
echo "OK: todos los contenedores de Vault están Ready."

## 2. PVC de auditoría

Comprueba que existe un PVC de auditoría por réplica, que todos están en estado `Bound` y muestra capacidad, StorageClass y volumen asociado.

In [ ]:
%%bash
set -euo pipefail

replicas=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get statefulset "${VAULT_HELM_RELEASE_NAME}" -o jsonpath='{.spec.replicas}')
audit_pvcs=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get pvc -o json | jq --arg release "${VAULT_HELM_RELEASE_NAME}" \
  '[.items[] | select(.metadata.name | startswith("audit-" + $release + "-"))]')

echo "${audit_pvcs}" | jq -r \
  '["PVC","STATUS","CAPACITY","STORAGE_CLASS","VOLUME"],
   (.[] | [.metadata.name,.status.phase,.status.capacity.storage,.spec.storageClassName,.spec.volumeName]) | @tsv' | column -t

count=$(jq 'length' <<<"${audit_pvcs}")
bound=$(jq '[.[] | select(.status.phase == "Bound")] | length' <<<"${audit_pvcs}")
test "${count}" -eq "${replicas}"
test "${bound}" -eq "${replicas}"
echo "OK: ${bound}/${replicas} PVC de auditoría están Bound."

## 3. Volúmenes, montajes y process namespace

Valida que Vault y el sidecar comparten `/vault/audit`, que el ConfigMap se monta como `/etc/logrotate.conf` y que `shareProcessNamespace` permite al sidecar enviar `SIGHUP` a Vault.

In [ ]:
%%bash
set -euo pipefail

sts=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get statefulset "${VAULT_HELM_RELEASE_NAME}" -o json)

jq -r '.spec.template.spec.containers[] as $c |
  $c.volumeMounts[]? |
  select(.mountPath == "/vault/audit" or .mountPath == "/etc/logrotate.conf") |
  [$c.name,.name,.mountPath,(.subPath // "-")] | @tsv' <<<"${sts}" | \
  { printf 'CONTAINER\tVOLUME\tMOUNT\tSUBPATH\n'; cat; } | column -t

jq -e '.spec.template.spec.shareProcessNamespace == true' <<<"${sts}" >/dev/null
jq -e '[.spec.template.spec.containers[] | select(.name == "vault") |
  .volumeMounts[] | select(.name == "audit" and .mountPath == "/vault/audit")] | length == 1' <<<"${sts}" >/dev/null
jq -e '[.spec.template.spec.containers[] | select(.name == "auditlog-rotator") |
  .volumeMounts[] | select(.name == "audit" and .mountPath == "/vault/audit")] | length == 1' <<<"${sts}" >/dev/null
jq -e '[.spec.template.spec.containers[] | select(.name == "auditlog-rotator") |
  .volumeMounts[] | select(.name == "logrotate-config" and .mountPath == "/etc/logrotate.conf")] | length == 1' <<<"${sts}" >/dev/null
echo "OK: montajes y process namespace configurados correctamente."

## 4. Configuración y procesos de logrotate

Muestra la política efectiva, la planificación del sidecar, sus procesos y sus últimos logs. También ejecuta `logrotate` en modo debug, que no modifica los ficheros.

In [ ]:
%%bash
set -euo pipefail

pod="${VAULT_HELM_RELEASE_NAME}-0"
echo '--- /etc/logrotate.conf ---'
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c auditlog-rotator -- cat /etc/logrotate.conf
echo '--- CRONTAB y procesos ---'
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c auditlog-rotator -- sh -c 'echo "CRONTAB=${CRONTAB:-unset}"; ps'
echo '--- logrotate debug (sin cambios) ---'
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c auditlog-rotator -- logrotate -d /etc/logrotate.conf
echo '--- logs del sidecar ---'
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  logs "${pod}" -c auditlog-rotator --tail=50
echo 'OK: configuración de logrotate legible y válida.'

## 5. Audit device de Vault

Consulta `sys/audit` desde `vaultpr-0` y confirma que existe un audit device de tipo `file` apuntando a `/vault/audit/vault.log`. La autenticación utiliza el usuario `tester` y requiere `VAULT_PR_TESTER_PASSWORD` en el fichero `.env`; el token secundario solo vive durante la celda.

In [ ]:
%%bash
set -euo pipefail
: "${VAULT_PR_TESTER_PASSWORD:?VAULT_PR_TESTER_PASSWORD no está definido; añádelo al fichero .env}"

pod="${VAULT_HELM_RELEASE_NAME}-0"
tester_login=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c vault -- env VAULT_SKIP_VERIFY=true \
  vault login -format=json -method=userpass username=tester password="${VAULT_PR_TESTER_PASSWORD}")
secondary_token=$(jq -r '.auth.client_token' <<<"${tester_login}")
unset tester_login
audit_devices=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c vault -- env VAULT_SKIP_VERIFY=true VAULT_TOKEN="${secondary_token}" \
  vault audit list -format=json)
unset secondary_token

jq -r 'to_entries[] | [.key,.value.type,(.value.options.file_path // "-")] | @tsv' \
  <<<"${audit_devices}" | { printf 'PATH\tTYPE\tFILE\n'; cat; } | column -t
jq -e --arg path "${VAULT_AUDIT_PATH}" \
  'to_entries | any(.value.type == "file" and .value.options.file_path == $path)' \
  <<<"${audit_devices}" >/dev/null
echo "OK: audit device file activo en ${VAULT_AUDIT_PATH}."

## 6. Ficheros de auditoría en todas las réplicas

Comprueba existencia, permisos, tamaño y sintaxis JSON. Solo muestra campos operativos de las últimas entradas para evitar volcar el registro de auditoría completo.

In [ ]:
%%bash
set -euo pipefail

pods=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get pods -l 'app.kubernetes.io/name=vault,component=server' -o jsonpath='{.items[*].metadata.name}')
for pod in ${pods}; do
  echo "--- ${pod} ---"
  kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
    exec "${pod}" -c vault -- sh -ec 'test -s /vault/audit/vault.log; ls -lh /vault/audit/vault.log*'
  recent=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
    exec "${pod}" -c vault -- tail -n 20 /vault/audit/vault.log)
  jq -e . >/dev/null <<<"${recent}"
  tail -n 3 <<<"${recent}" | \
    jq -r '[.time,.type,(.request.operation // "-"),(.request.path // "-")] | @tsv'
done
echo 'OK: los audit logs existen y las últimas entradas contienen JSON válido.'

## 7. Persistencia asociada al pod

Relaciona cada pod con su PVC de auditoría y muestra el uso real del filesystem desde el contenedor Vault.

In [ ]:
%%bash
set -euo pipefail

for pod in $(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get pods -l 'app.kubernetes.io/name=vault,component=server' -o name | cut -d/ -f2); do
  claim=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" get pod "${pod}" \
    -o json | jq -r '.spec.volumes[] | select(.name == "audit") | .persistentVolumeClaim.claimName')
  printf '%s -> %s\n' "${pod}" "${claim}"
  kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
    exec "${pod}" -c vault -- df -h /vault/audit
done

## 8. Prueba funcional opcional de rotación

Al habilitarla, fuerza una rotación en `vaultpr-0`, genera una operación auditada y confirma que existen tanto el fichero rotado como el nuevo fichero activo. Esta prueba modifica únicamente los ficheros de auditoría conforme a la política instalada.

In [ ]:
%%bash
set -euo pipefail

FORCE_ROTATION=0
if [[ "${FORCE_ROTATION}" != "1" ]]; then
  echo 'Prueba omitida. Cambia FORCE_ROTATION=1 para ejecutarla.'
  exit 0
fi
: "${VAULT_PR_TESTER_PASSWORD:?VAULT_PR_TESTER_PASSWORD no está definido; añádelo al fichero .env}"

pod="${VAULT_HELM_RELEASE_NAME}-0"
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c auditlog-rotator -- \
  logrotate -s /tmp/logrotate-force.status -f /etc/logrotate.conf
tester_login=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c vault -- env VAULT_SKIP_VERIFY=true \
  vault login -format=json -method=userpass username=tester password="${VAULT_PR_TESTER_PASSWORD}")
secondary_token=$(jq -r '.auth.client_token' <<<"${tester_login}")
unset tester_login
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c vault -- env VAULT_SKIP_VERIFY=true VAULT_TOKEN="${secondary_token}" \
  vault token lookup -format=json >/dev/null
unset secondary_token

kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c vault -- sh -ec '
    test -s /vault/audit/vault.log
    find /vault/audit -maxdepth 1 -type f -name "vault.log.*" | grep -q .
    ls -lh /vault/audit/vault.log*
  '
last_entry=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c vault -- tail -n 1 /vault/audit/vault.log)
jq -e . >/dev/null <<<"${last_entry}"
echo 'OK: rotación forzada, reapertura mediante SIGHUP y nueva escritura validadas.'